# EDA — gráficos direto no Glue Studio Notebook, a partir da Gold

Este notebook roda a análise exploratória **dentro da AWS** (Glue Notebook,
Spark), lendo as 7 tabelas Gold e desenhando os mesmos gráficos que o
protótipo local (`src/eda_charts.py`) gerou a partir dos mesmos dados —
assim dá pra conferir, ao vivo, que os números batem.

Cada célula usa a magic `%matplot plt` do Glue para mostrar o gráfico direto
na saída da célula (Spark → `.toPandas()` nos resultados já agregados da
Gold, que são pequenos → matplotlib normal).

In [ ]:
%idle_timeout 2880
%glue_version 4.0
%worker_type G.1X
%number_of_workers 5


In [ ]:
%%configure
{
  "--enable-glue-datacatalog": "true"
}


In [ ]:
import sys
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
import pyspark.sql.functions as F
import pyspark.sql.types as T

sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)


In [ ]:
# =============================================================================
# CONFIGURACAO — valores conferidos AO VIVO no console AWS do time em
# 2026-10-18 (Glue Data Catalog + S3). Troque so se os nomes mudarem.
#
# Os 3 CSVs brutos ficam juntos, soltos (sem subpasta por edicao), direto em
# s3://tech-challenge-gt26/bronze/ — e com nomes de arquivo que NAO indicam a
# edicao de forma confiavel. Mapeamento confirmado por inspecao de conteudo
# (S3 Select: formato do cabecalho + pergunta "passou por layoff em <ano>"):
#   - "State_of_data_2022.csv"                      -> edicao 2023-2024
#   - "State_of_data_BR_2023_Kaggle - df_survey_2023.csv" -> edicao 2024-2025
#   - "Final Dataset - State of Data 2025-2026 - Kaggle.csv" -> edicao 2025-2026
#
# Os nomes de tabela abaixo (tb_bronze_*, tb_silver_*, tb_gold_*) usam um
# prefixo "tb_" de proposito para conviver, dentro do MESMO database
# db_techchallenge_gt_fiap, com as tabelas "bronze"/"silver"/"gold" que ja
# existem — nada aqui sobrescreve ou apaga o que ja foi feito.
# =============================================================================
BUCKET = "s3://tech-challenge-gt26"

RAW_PATHS = {
    "2023-2024": f"{BUCKET}/bronze/State_of_data_2022.csv",
    "2024-2025": f"{BUCKET}/bronze/State_of_data_BR_2023_Kaggle - df_survey_2023.csv",
    "2025-2026": f"{BUCKET}/bronze/Final Dataset - State of Data 2025-2026 - Kaggle.csv",
}

# Prefixo NOVO e distinto do "bronze/" (lowercase) onde os CSVs brutos ja
# estao — evita qualquer risco de sobrescrever a fonte original.
BRONZE_PATH = f"{BUCKET}/bronze_parquet/"
SILVER_PATH = f"{BUCKET}/Silver/"
GOLD_PATH = f"{BUCKET}/Gold/"

DB_BRONZE = "db_techchallenge_gt_fiap"
DB_SILVER = "db_techchallenge_gt_fiap"
DB_GOLD = "db_techchallenge_gt_fiap"

for db in (DB_BRONZE, DB_SILVER, DB_GOLD):
    spark.sql(f"CREATE DATABASE IF NOT EXISTS {db}")


In [ ]:
import matplotlib.pyplot as plt

MAGENTA = "#ED145B"
AZUL = "#2A78D6"
VERDE = "#008300"
AMBAR = "#EDA100"
VIOLETA = "#6250D6"
SENIORIDADE_ORDEM = ["Júnior", "Pleno", "Sênior", "Especialista/Staff+"]

plt.rcParams.update({"figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb"})


## Q1 — Estrutura do mercado: top 10 cargos

In [ ]:
perfil = spark.table(f"{DB_GOLD}.gold_perfil_mercado").toPandas()
top_cargos = perfil.groupby("cargo_atual")["n_respondentes"].sum().sort_values().tail(10)

plt.figure(figsize=(8, 5))
plt.barh(top_cargos.index, top_cargos.values, color=MAGENTA)
plt.title("Os 10 cargos mais comuns no mercado de Dados (BR)")
plt.xlabel("Nº de respondentes (2023-2026)")
plt.tight_layout()
%matplot plt


## Q2 — Perfis mais valorizados: salário médio por cargo

In [ ]:
remuneracao = spark.table(f"{DB_GOLD}.gold_remuneracao").toPandas()
remuneracao["peso"] = remuneracao["salario_medio"] * remuneracao["n_respondentes"]
por_cargo = remuneracao.groupby("cargo_atual").agg(peso=("peso", "sum"), n=("n_respondentes", "sum"))
por_cargo["salario_medio"] = por_cargo["peso"] / por_cargo["n"]
top_pagos = por_cargo[por_cargo["n"] >= 50].sort_values("salario_medio").tail(10)

plt.figure(figsize=(8, 5))
plt.barh(top_pagos.index, top_pagos["salario_medio"], color=MAGENTA)
plt.title("Remuneração média por cargo (entre os mais comuns)")
plt.xlabel("Salário médio estimado (R$/mês)")
plt.tight_layout()
%matplot plt


## Q3 — Diversidade de gênero: participação por senioridade e gap salarial

In [ ]:
import numpy as np

diversidade = spark.table(f"{DB_GOLD}.gold_diversidade_genero").toPandas()
ultimo_ano = diversidade["ano_pesquisa"].max()
dg = diversidade[(diversidade["ano_pesquisa"] == ultimo_ano) & (diversidade["genero"].isin(["Masculino", "Feminino"]))]
pivot_dg = dg.pivot(index="nivel_senioridade", columns="genero", values="pct_dentro_do_nivel").reindex(SENIORIDADE_ORDEM)

x = np.arange(len(pivot_dg.index))
largura = 0.35
plt.figure(figsize=(7.5, 4.5))
plt.bar(x - largura/2, pivot_dg["Masculino"], largura, label="Masculino", color=MAGENTA)
plt.bar(x + largura/2, pivot_dg["Feminino"], largura, label="Feminino", color=AZUL)
plt.xticks(x, pivot_dg.index)
plt.ylabel("% dentro do nível")
plt.title(f"Participação por gênero e senioridade ({ultimo_ano})")
plt.legend()
plt.tight_layout()
%matplot plt


In [ ]:
sal_genero = remuneracao[remuneracao["genero"].isin(["Masculino", "Feminino"])]
sal_genero = sal_genero.groupby(["nivel_senioridade", "genero"]).apply(
    lambda g: (g["salario_medio"] * g["n_respondentes"]).sum() / g["n_respondentes"].sum()
).unstack().reindex(SENIORIDADE_ORDEM)

plt.figure(figsize=(7.5, 4.5))
plt.bar(x - largura/2, sal_genero["Masculino"], largura, label="Masculino", color=MAGENTA)
plt.bar(x + largura/2, sal_genero["Feminino"], largura, label="Feminino", color=AZUL)
plt.xticks(x, sal_genero.index)
plt.ylabel("Salário médio estimado (R$/mês)")
plt.title("Diferença salarial entre gêneros, por senioridade")
plt.legend()
plt.tight_layout()
%matplot plt


## Q4 — Tecnologias mais adotadas (edição mais recente)

In [ ]:
tecnologias = spark.table(f"{DB_GOLD}.gold_tecnologias_adocao").toPandas()
ultimo_ano_tec = tecnologias["ano_pesquisa"].max()
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
categorias_labels = {
    "linguagem": "Linguagens de programação",
    "banco_dados": "Bancos de dados",
    "cloud": "Provedores de Cloud",
    "ferramenta_bi": "Ferramentas de BI",
}
for ax, (cat, label) in zip(axes.flat, categorias_labels.items()):
    top = (
        tecnologias[(tecnologias["categoria_tecnologia"] == cat) & (tecnologias["ano_pesquisa"] == ultimo_ano_tec)]
        .nlargest(6, "pct_adocao").sort_values("pct_adocao")
    )
    ax.barh(top["tecnologia"].tolist(), top["pct_adocao"], color=MAGENTA)
    ax.set_title(label, fontsize=12, fontweight="bold")
fig.suptitle(f"Tecnologias mais adotadas ({ultimo_ano_tec})", fontsize=14, fontweight="bold")
plt.tight_layout()
%matplot plt


## Q5 — Adoção de IA Generativa e prioridade executiva

In [ ]:
adocao_ia = spark.table(f"{DB_GOLD}.gold_adocao_ia").toPandas().sort_values("ano_pesquisa")

plt.figure(figsize=(7, 4.5))
plt.plot(adocao_ia["ano_pesquisa"], adocao_ia["pct_usa_ia_generativa"], marker="o", linewidth=2.5, markersize=8, color=MAGENTA)
for _, row in adocao_ia.iterrows():
    plt.text(row["ano_pesquisa"], row["pct_usa_ia_generativa"] + 1.5, f"{row['pct_usa_ia_generativa']:.0f}%", ha="center")
plt.ylim(0, 105)
plt.title("Adoção individual de IA Generativa dispara")
plt.ylabel("% que usa IA Generativa no trabalho")
plt.tight_layout()
%matplot plt


In [ ]:
prioridade_ia = spark.table(f"{DB_GOLD}.gold_prioridade_ia_empresa").toPandas()
ultimo_ano_ia = prioridade_ia["ano_pesquisa"].max()
ordem_prioridade = ["Não é prioridade", "Não sabe opinar", "Iniciativa isolada / pouco foco",
                    "Prioridade de médio prazo", "Prioridade alta (foco executivo)"]
pri = prioridade_ia[prioridade_ia["ano_pesquisa"] == ultimo_ano_ia].set_index("prioridade_ia_empresa").reindex(ordem_prioridade)

plt.figure(figsize=(7, 4.5))
plt.barh(pri.index, pri["pct_gestores"], color=[AMBAR, AMBAR, VIOLETA, AZUL, MAGENTA])
plt.xlabel("% dos gestores")
plt.title(f"Prioridade de IA Generativa nas empresas ({ultimo_ano_ia})")
plt.tight_layout()
%matplot plt


## Q6 — Diferenças por região e modelo de trabalho

In [ ]:
sal_regiao = remuneracao.groupby("regiao").apply(
    lambda g: (g["salario_medio"] * g["n_respondentes"]).sum() / g["n_respondentes"].sum()
).sort_values()

plt.figure(figsize=(7, 4.5))
plt.barh(sal_regiao.index, sal_regiao.values, color=MAGENTA)
plt.xlabel("Salário médio estimado (R$/mês)")
plt.title("Salário médio por região")
plt.tight_layout()
%matplot plt


## Conferindo com o resultado local

Compare os números acima com `docs/_eda_achados.txt` e os PNGs em
`presentation/charts/` — devem bater (mesma base, mesma lógica de
harmonização, só rodando em Spark/AWS em vez de pandas local).